# OTDR event classification — training and validation

Этот notebook обучает классификаторы локального OTDR-события на трёх ручных классах: `bend`, `connector`, `break`.

- Train: `train_gt_events.csv`
- Validation: `val_gt_events.csv`
- Split уже выполнен на уровне файлов, поэтому трассы validation не попадают в обучение.
- Модели: Decision Tree, Random Forest, Extra Trees, Gradient Boosting, Logistic Regression и SVM-RBF.
- Выбор лучшей модели: по `macro F1` на validation.

In [ ]:
from pathlib import Path
import json
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.base import clone
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay

RANDOM_STATE = 42
CLASSES = ['bend', 'connector', 'break']

FEATURE_COLS = [
    'm_norm', 'db_at_event', 'local_mean_db', 'local_std_db',
    'pre_mean_db', 'post_mean_db', 'loss_dB', 'peak_above_bg_dB',
    'pre_slope_dB_per_km', 'post_slope_dB_per_km', 'slope_change_dB_per_km',
    'derivative_at_event_dB_per_km', 'max_pre_derivative', 'min_post_derivative',
    'pre_std_db', 'post_std_db', 'post_to_pre_std_ratio',
    'peak_width_m', 'local_range_db'
]

DATA_DIR = Path.cwd() / 'gt_event_model_data'
TRAIN_CSV = DATA_DIR / 'train_gt_events.csv'
VAL_CSV = DATA_DIR / 'val_gt_events.csv'
OUT_DIR = DATA_DIR / 'model_results'
OUT_DIR.mkdir(parents=True, exist_ok=True)

print('Data folder:', DATA_DIR)
print('Output folder:', OUT_DIR)

In [ ]:
train_df = pd.read_csv(TRAIN_CSV)
val_df = pd.read_csv(VAL_CSV)

missing_train = [c for c in FEATURE_COLS if c not in train_df.columns]
missing_val = [c for c in FEATURE_COLS if c not in val_df.columns]
if missing_train or missing_val:
    raise ValueError(f'Missing features. train={missing_train}, val={missing_val}')

train_df = train_df[train_df['label'].isin(CLASSES)].dropna(subset=FEATURE_COLS).copy()
val_df = val_df[val_df['label'].isin(CLASSES)].dropna(subset=FEATURE_COLS).copy()

X_train = train_df[FEATURE_COLS]
y_train = train_df['label']
X_val = val_df[FEATURE_COLS]
y_val = val_df['label']

print('Train shape:', X_train.shape)
print(y_train.value_counts().reindex(CLASSES, fill_value=0))
print('\nValidation shape:', X_val.shape)
print(y_val.value_counts().reindex(CLASSES, fill_value=0))

assert set(train_df['file']).isdisjoint(set(val_df['file'])), 'Data leakage: same file in train and validation!'
print('\nOK: file-level split is clean; no filename appears in both train and validation.')

In [ ]:
models = {
    'DecisionTree': DecisionTreeClassifier(
        max_depth=8, min_samples_leaf=3, class_weight='balanced', random_state=RANDOM_STATE
    ),
    'RandomForest': RandomForestClassifier(
        n_estimators=500, max_depth=12, min_samples_leaf=2,
        class_weight='balanced', random_state=RANDOM_STATE, n_jobs=-1
    ),
    'ExtraTrees': ExtraTreesClassifier(
        n_estimators=500, max_depth=14, min_samples_leaf=2,
        class_weight='balanced', random_state=RANDOM_STATE, n_jobs=-1
    ),
    'GradientBoosting': GradientBoostingClassifier(
        n_estimators=250, learning_rate=0.05, max_depth=3, random_state=RANDOM_STATE
    ),
    'LogisticRegression': Pipeline([
        ('scaler', StandardScaler()),
        ('clf', LogisticRegression(max_iter=3000, class_weight='balanced', random_state=RANDOM_STATE))
    ]),
    'SVM_RBF': Pipeline([
        ('scaler', StandardScaler()),
        ('clf', SVC(kernel='rbf', C=2.0, gamma='scale', class_weight='balanced',
                    probability=True, random_state=RANDOM_STATE))
    ]),
}

min_class_n = int(y_train.value_counts().min())
n_splits = min(5, min_class_n)
cv = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=RANDOM_STATE)
print(f'CV folds: {n_splits}; smallest training class: {min_class_n}')

In [ ]:
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

results = []
fitted_models = {}
val_predictions = {}

for name, model in models.items():
    cv_scores = cross_val_score(model, X_train, y_train, cv=cv, scoring='f1_macro', n_jobs=-1)

    fitted = clone(model).fit(X_train, y_train)
    pred = fitted.predict(X_val)
    val_predictions[name] = pred
    fitted_models[name] = fitted

    p_macro, r_macro, f1_macro, _ = precision_recall_fscore_support(
        y_val, pred, labels=CLASSES, average='macro', zero_division=0
    )
    _, _, f1_weighted, _ = precision_recall_fscore_support(
        y_val, pred, labels=CLASSES, average='weighted', zero_division=0
    )

    per_p, per_r, per_f1, per_support = precision_recall_fscore_support(
        y_val, pred, labels=CLASSES, zero_division=0
    )

    row = {
        'model': name,
        'cv_f1_macro_train_mean': cv_scores.mean(),
        'cv_f1_macro_train_std': cv_scores.std(),
        'val_accuracy': accuracy_score(y_val, pred),
        'val_f1_macro': f1_macro,
        'val_f1_weighted': f1_weighted,
        'val_precision_macro': p_macro,
        'val_recall_macro': r_macro,
    }
    for cls, precision, recall, f1, support in zip(CLASSES, per_p, per_r, per_f1, per_support):
        row[f'{cls}_precision'] = precision
        row[f'{cls}_recall'] = recall
        row[f'{cls}_f1'] = f1
        row[f'{cls}_support'] = support
    results.append(row)

comparison = pd.DataFrame(results).sort_values('val_f1_macro', ascending=False).reset_index(drop=True)
comparison.round(4)

In [ ]:
comparison.to_csv(OUT_DIR / 'model_comparison.csv', index=False)

for name in comparison['model']:
    pred = val_predictions[name]
    print('=' * 80)
    print(name)
    print(classification_report(y_val, pred, labels=CLASSES, digits=3, zero_division=0))

    fig, ax = plt.subplots(figsize=(5, 4))
    ConfusionMatrixDisplay(confusion_matrix(y_val, pred, labels=CLASSES), display_labels=CLASSES).plot(
        ax=ax, cmap='Blues', values_format='d', colorbar=False
    )
    ax.set_title(f'{name} — validation confusion matrix')
    plt.tight_layout()
    plt.show()

In [ ]:
best_name = comparison.iloc[0]['model']
best_model = fitted_models[best_name]

bundle = {
    'model': best_model,
    'model_name': best_name,
    'features': FEATURE_COLS,
    'classes': CLASSES,
    'train_rows': len(train_df),
    'val_rows': len(val_df),
    'split': 'file-level; manual JSON annotations only',
    'best_val_f1_macro': float(comparison.iloc[0]['val_f1_macro']),
}

model_path = OUT_DIR / 'best_gt_event_classifier.joblib'
joblib.dump(bundle, model_path)

print(f'Best model: {best_name}')
print(f'Validation macro F1: {comparison.iloc[0]["val_f1_macro"]:.4f}')
print(f'Saved model: {model_path}')
print(f'Saved comparison: {OUT_DIR / "model_comparison.csv"}')

In [ ]:
# Feature importance доступна для tree-based моделей.
if best_name in {'DecisionTree', 'RandomForest', 'ExtraTrees', 'GradientBoosting'}:
    estimator = best_model
    importance = pd.Series(estimator.feature_importances_, index=FEATURE_COLS).sort_values(ascending=False)
    display(importance.to_frame('importance'))

    fig, ax = plt.subplots(figsize=(9, 6))
    importance.sort_values().plot.barh(ax=ax, color='#1f77b4')
    ax.set_title(f'{best_name}: feature importances')
    ax.set_xlabel('importance')
    plt.tight_layout()
    plt.show()
else:
    print(f'Feature importances are not directly available for {best_name}.')